# Notebook 02 — Preprocessing

**Goal:** Clean and standardize both CSV datasets.

**Actual schema of `all_marathons_final.csv`:**
```
index, Name, Sex, Age, Gun Time, Place, Sex Place, Div Place, Sex1, Agegroup,
Time, Country, data, Year, Marathon
```
- `Time` is already in float minutes — no parsing needed
- `Sex` is already M/F
- `Country` + `data` are messy city/state strings, not true nationality
- `Marathon` contains the race name (e.g. `Twin_Cities`, `Boston`)
- Only `Sex, Age, Time, Year, Marathon` are loaded — the other columns
  (names, places, hometown strings) are unused downstream and waste RAM

In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR  = '../data/raw'
PROC_DIR = '../data/processed'
os.makedirs(PROC_DIR, exist_ok=True)

MARATHON_CSV = os.path.join(RAW_DIR, 'marathon', 'all_marathons_final.csv')
ATHLETES_CSV = os.path.join(RAW_DIR, 'athletes', 'run_ww_2019_w.csv')

## 1. Load Raw Data

In [2]:
# Load only the columns the pipeline uses, with compact dtypes — the full
# files don't fit comfortably in a small machine's RAM.
# (on_bad_lines='skip' is a precaution; the current file has no malformed rows)
MARATHON_USECOLS = ['Sex', 'Age', 'Time', 'Year', 'Marathon']
MARATHON_DTYPES  = {'Sex': 'category', 'Age': 'float32',
                    'Time': 'float32', 'Marathon': 'category'}

marathon_raw = pd.read_csv(MARATHON_CSV, usecols=MARATHON_USECOLS,
                           dtype=MARATHON_DTYPES, on_bad_lines='skip')
print(f'Marathon raw: {marathon_raw.shape} — '
      f'{marathon_raw.memory_usage(deep=True).sum()/1e6:.0f} MB in RAM')

# Only the weekly running file — the m/q files duplicate it at coarser
# granularity and the covid-* files have a different schema.
ATHLETES_USECOLS = ['athlete', 'gender', 'age_group', 'country', 'major', 'distance']
ATHLETES_DTYPES  = {'athlete': 'int32', 'gender': 'category', 'age_group': 'category',
                    'country': 'category', 'major': 'category', 'distance': 'float32'}

if os.path.exists(ATHLETES_CSV):
    athletes_raw = pd.read_csv(ATHLETES_CSV, usecols=ATHLETES_USECOLS, dtype=ATHLETES_DTYPES)
    print(f'Athletes raw: {athletes_raw.shape} — '
          f'{athletes_raw.memory_usage(deep=True).sum()/1e6:.0f} MB in RAM')
else:
    print('WARNING: run_ww_2019_w.csv not found — run the download cell in notebook 01 first.')
    athletes_raw = pd.DataFrame()  # empty placeholder

Marathon raw: (3343216, 5) — 60 MB in RAM


Athletes raw: (1893424, 6) — 27 MB in RAM


## 2. Rename Marathon Columns to Canonical Names

In [3]:
# Confirmed actual column names from Zenodo all_marathons_final.csv
MARATHON_RENAME = {
    'Name':      'name',
    'Sex':       'gender',       # already M/F
    'Age':       'age',
    'Gun Time':  'gun_time_str', # HH:MM:SS format
    'Time':      'finish_min',   # ALREADY in float minutes
    'Place':     'overall_place',
    'Sex Place': 'sex_place',
    'Div Place': 'div_place',
    'Agegroup':  'age_group_raw',
    'Year':      'year',
    'Marathon':  'race',
    # 'Country' and 'data' are city/state strings, not true nationality
}

marathon = marathon_raw.rename(
    columns={k: v for k, v in MARATHON_RENAME.items() if k in marathon_raw.columns}
)
# Drop redundant Sex1 column (duplicate of gender)
marathon = marathon.drop(columns=['Sex1'], errors='ignore')

print('Marathon columns after rename:', list(marathon.columns))

Marathon columns after rename: ['gender', 'age', 'finish_min', 'year', 'race']


In [4]:
# Aggregate the weekly Strava rows (~52 per athlete) to one row per athlete.
# weekly_km = mean distance per week
if not athletes_raw.empty:
    athletes = (
        athletes_raw
        .groupby('athlete', as_index=False)
        .agg(
            gender=('gender', 'first'),
            age_group=('age_group', 'first'),
            country=('country', 'first'),
            major=('major', 'first'),
            weekly_km=('distance', 'mean'),
        )
    )
    print(f'Athletes: {len(athletes):,} (one row per athlete)')
    print('Columns:', list(athletes.columns))
else:
    athletes = pd.DataFrame()
    print('Athlete dataset not available.')

Athletes: 36,412 (one row per athlete)
Columns: ['athlete', 'gender', 'age_group', 'country', 'major', 'weekly_km']


## 3. Filter Outliers — Marathon Results

In [5]:
n_before = len(marathon)

# Finish time bounds (already in minutes)
FINISH_MIN_LOW  = 120   # ~2h — world record is ~2:01
FINISH_MIN_HIGH = 600   # 10h — generous upper bound for walkers

marathon['finish_min'] = pd.to_numeric(marathon['finish_min'], errors='coerce')
marathon = marathon[
    marathon['finish_min'].between(FINISH_MIN_LOW, FINISH_MIN_HIGH)
].copy()

# Age bounds
marathon['age'] = pd.to_numeric(marathon['age'], errors='coerce')
marathon = marathon[marathon['age'].between(15, 85)].copy()

# Remove rows without gender
marathon = marathon[marathon['gender'].isin(['M', 'F'])].copy()

print(f'Rows removed: {n_before - len(marathon):,}  ({(n_before - len(marathon))/n_before*100:.1f}%)')
print(f'Rows remaining: {len(marathon):,}')
print(f'Gender split: {marathon["gender"].value_counts().to_dict()}')

Rows removed: 488,943  (14.6%)
Rows remaining: 2,854,273
Gender split: {'M': 1700597, 'F': 1153676}


## 4. Map Race Names to Clean City Names

The `race` column contains marathon names like `Twin_Cities`, `Boston`. We map these to the host city used for weather API lookups.

In [6]:
print('Unique race names in dataset:')
print(sorted(marathon['race'].dropna().unique().tolist()))

Unique race names in dataset:
['California_International', 'Chicago', 'Grandmas', 'Honolulu', 'Houston', 'LA', 'MarineCorps', 'NYC', 'Philadelphia', 'Twin_Cities']


In [7]:
# Map marathon name → host city (used for the weather API lookups)
RACE_TO_CITY = {
    'Boston':          'Boston',
    'Chicago':         'Chicago',
    'NYC':             'New York City',
    'New_York':        'New York City',
    'New York':        'New York City',
    'Twin_Cities':     'Minneapolis',
    'Twin Cities':     'Minneapolis',
    'Grandmas':        'Duluth',
    "Grandma's":       'Duluth',
    'Houston':         'Houston',
    'LA':              'Los Angeles',
    'Los_Angeles':     'Los Angeles',
    'Los Angeles':     'Los Angeles',
    'Marine_Corps':    'Washington',
    'Marine Corps':    'Washington',
    'Big_Sur':         'Monterey',
    'Big Sur':         'Monterey',
}

marathon['city_clean'] = marathon['race'].map(RACE_TO_CITY)
unmapped = marathon[marathon['city_clean'].isna()]['race'].unique()
if len(unmapped):
    print(f'WARNING: {len(unmapped)} race name(s) not mapped to a city: {unmapped}')
    print('Add them to RACE_TO_CITY above.')
else:
    print('All race names mapped. City distribution:')
    print(marathon['city_clean'].value_counts())

Categories (10, str): ['Twin_Cities', 'Grandmas', 'Houston', 'Philadelphia', ..., 'California_International', 'NYC', 'LA', 'Chicago']
Add them to RACE_TO_CITY above.


## 5. Add Race Date Column

In [8]:
RACE_DATES = {
    ('Boston', 2000): '2000-04-17', ('Boston', 2001): '2001-04-16',
    ('Boston', 2002): '2002-04-15', ('Boston', 2003): '2003-04-21',
    ('Boston', 2004): '2004-04-19', ('Boston', 2005): '2005-04-18',
    ('Boston', 2006): '2006-04-17', ('Boston', 2007): '2007-04-16',
    ('Boston', 2008): '2008-04-21', ('Boston', 2009): '2009-04-20',
    ('Boston', 2010): '2010-04-19', ('Boston', 2011): '2011-04-18',
    ('Boston', 2012): '2012-04-16', ('Boston', 2013): '2013-04-15',
    ('Boston', 2014): '2014-04-21', ('Boston', 2015): '2015-04-20',
    ('Boston', 2016): '2016-04-18', ('Boston', 2017): '2017-04-17',
    ('Boston', 2018): '2018-04-16', ('Boston', 2019): '2019-04-15',
    ('Chicago', 2000): '2000-10-22', ('Chicago', 2001): '2001-10-07',
    ('Chicago', 2002): '2002-10-13', ('Chicago', 2003): '2003-10-12',
    ('Chicago', 2004): '2004-10-10', ('Chicago', 2005): '2005-10-09',
    ('Chicago', 2006): '2006-10-22', ('Chicago', 2007): '2007-10-07',
    ('Chicago', 2008): '2008-10-12', ('Chicago', 2009): '2009-10-11',
    ('Chicago', 2010): '2010-10-10', ('Chicago', 2011): '2011-10-09',
    ('Chicago', 2012): '2012-10-07', ('Chicago', 2013): '2013-10-13',
    ('Chicago', 2014): '2014-10-12', ('Chicago', 2015): '2015-10-11',
    ('Chicago', 2016): '2016-10-09', ('Chicago', 2017): '2017-10-08',
    ('Chicago', 2018): '2018-10-07', ('Chicago', 2019): '2019-10-13',
    ('New York City', 2000): '2000-11-05', ('New York City', 2001): '2001-11-04',
    ('New York City', 2002): '2002-11-03', ('New York City', 2003): '2003-11-02',
    ('New York City', 2004): '2004-11-07', ('New York City', 2005): '2005-11-06',
    ('New York City', 2006): '2006-11-05', ('New York City', 2007): '2007-11-04',
    ('New York City', 2008): '2008-11-02', ('New York City', 2009): '2009-11-01',
    ('New York City', 2010): '2010-11-07', ('New York City', 2011): '2011-11-06',
    ('New York City', 2013): '2013-11-03', ('New York City', 2014): '2014-11-02',
    ('New York City', 2015): '2015-11-01', ('New York City', 2016): '2016-11-06',
    ('New York City', 2017): '2017-11-05', ('New York City', 2018): '2018-11-04',
    ('New York City', 2019): '2019-11-03',
    ('Minneapolis', 2000): '2000-10-01', ('Minneapolis', 2001): '2001-10-07',
    ('Minneapolis', 2002): '2002-10-06', ('Minneapolis', 2003): '2003-10-05',
    ('Minneapolis', 2004): '2004-10-03', ('Minneapolis', 2005): '2005-10-02',
    ('Minneapolis', 2006): '2006-10-01', ('Minneapolis', 2007): '2007-10-07',
    ('Minneapolis', 2008): '2008-10-05', ('Minneapolis', 2009): '2009-10-04',
    ('Minneapolis', 2010): '2010-10-03', ('Minneapolis', 2011): '2011-10-02',
    ('Minneapolis', 2012): '2012-10-07', ('Minneapolis', 2013): '2013-10-06',
    ('Minneapolis', 2014): '2014-10-05', ('Minneapolis', 2015): '2015-10-04',
    ('Minneapolis', 2016): '2016-10-02', ('Minneapolis', 2017): '2017-10-01',
    ('Minneapolis', 2018): '2018-10-07', ('Minneapolis', 2019): '2019-10-06',
    ('Houston', 2000): '2000-01-16', ('Houston', 2001): '2001-01-14',
    ('Houston', 2002): '2002-01-20', ('Houston', 2003): '2003-01-19',
    ('Houston', 2004): '2004-01-18', ('Houston', 2005): '2005-01-16',
    ('Houston', 2006): '2006-01-15', ('Houston', 2007): '2007-01-14',
    ('Houston', 2008): '2008-01-13', ('Houston', 2009): '2009-01-18',
    ('Houston', 2010): '2010-01-17', ('Houston', 2011): '2011-01-30',
    ('Houston', 2012): '2012-01-15', ('Houston', 2013): '2013-01-13',
    ('Houston', 2014): '2014-01-19', ('Houston', 2015): '2015-01-18',
    ('Houston', 2016): '2016-01-17', ('Houston', 2017): '2017-01-15',
    ('Houston', 2018): '2018-01-14', ('Houston', 2019): '2019-01-20',
    ('Duluth', 2000): '2000-06-17', ('Duluth', 2001): '2001-06-16',
    ('Duluth', 2002): '2002-06-22', ('Duluth', 2003): '2003-06-21',
    ('Duluth', 2004): '2004-06-19', ('Duluth', 2005): '2005-06-18',
    ('Duluth', 2006): '2006-06-17', ('Duluth', 2007): '2007-06-16',
    ('Duluth', 2008): '2008-06-21', ('Duluth', 2009): '2009-06-20',
    ('Duluth', 2010): '2010-06-19', ('Duluth', 2011): '2011-06-18',
    ('Duluth', 2012): '2012-06-16', ('Duluth', 2013): '2013-06-22',
    ('Duluth', 2014): '2014-06-21', ('Duluth', 2015): '2015-06-20',
    ('Duluth', 2016): '2016-06-18', ('Duluth', 2017): '2017-06-17',
    ('Duluth', 2018): '2018-06-16', ('Duluth', 2019): '2019-06-22',
}

marathon['year'] = pd.to_numeric(marathon['year'], errors='coerce').astype('Int64')
marathon['race_date'] = marathon.apply(
    lambda r: RACE_DATES.get((r['city_clean'], int(r['year'])) if pd.notna(r['year']) else None, np.nan),
    axis=1
)

covered = marathon['race_date'].notna().sum()
print(f'Rows with race_date: {covered:,} / {len(marathon):,}  ({covered/len(marathon)*100:.1f}%)')
print('\nDate coverage by city:')
print(marathon.groupby('city_clean')['race_date'].agg(['count', lambda x: x.notna().sum()]).rename(columns={'count': 'total', '<lambda_0>': 'with_date'}))

Rows with race_date: 1,740,722 / 2,854,273  (61.0%)

Date coverage by city:


                total  with_date
city_clean                      
Chicago        586767     586767
Duluth         114929     114929
Houston        106490     106490
Los Angeles         0          0
Minneapolis    131924     131924
New York City  800612     800612


## 6. Sanity-Check Strava Data

The weekly file was aggregated to one row per athlete in section 2 (`weekly_km` = mean km per week across 2019). Here we just confirm the per-athlete distribution looks plausible.

In [9]:
if not athletes.empty:
    print(f'Strava athletes: {len(athletes):,}')
    print(athletes['weekly_km'].describe())

Strava athletes: 36,412


count    36412.000000
mean        29.241417
std         22.979923
min          0.000385
25%         11.786166
50%         24.386907
75%         41.164400
max        218.049881
Name: weekly_km, dtype: float64


## 7. Save Cleaned Files

In [10]:
marathon_out = os.path.join(PROC_DIR, 'marathon_clean.csv')
marathon.to_csv(marathon_out, index=False)
print(f'Saved: {marathon_out}  — {len(marathon):,} rows × {marathon.shape[1]} cols')

if not athletes.empty:
    athletes_out = os.path.join(PROC_DIR, 'athletes_clean.csv')
    athletes.to_csv(athletes_out, index=False)
    print(f'Saved: {athletes_out} — {len(athletes):,} rows')
else:
    print('Athlete dataset not saved (not available).')

print('\nFinal marathon columns:')
for c in marathon.columns:
    print(f'  {c}: {marathon[c].dtype}  (nulls: {marathon[c].isna().sum():,})')

Saved: ../data/processed/marathon_clean.csv  — 2,854,273 rows × 7 cols


Saved: ../data/processed/athletes_clean.csv — 36,412 rows

Final marathon columns:
  gender: category  (nulls: 0)
  age: float32  (nulls: 0)
  finish_min: float32  (nulls: 0)
  year: Int64  (nulls: 0)
  race: category  (nulls: 0)
  city_clean: str  (nulls: 756,335)


  race_date: str  (nulls: 1,113,551)
